# 🤖 NanoSurfaces Prime — openBIS Chatbot

Ask questions about substances, molecules, crystals, instrument components, sample preparation workflows, project experiments, measurements, and publications.

> **Read-Only Guarantee**: This assistant only reads and cross-references data from openBIS. It cannot modify or delete records. All openBIS permIDs are clickable hyperlinks.

In [ ]:
%%capture
import os
import re
import time
import markdown
from pathlib import Path
from IPython.display import display, HTML, Javascript
import ipywidgets as ipw
import nest_asyncio
nest_asyncio.apply()

from src import utils
from src.ai_agent import OpenBISAgent

CONFIG_PATH = "config/config.json"
LLM_CONFIG_PATH = "/home/jovyan/api_keys/llm_config.json"
CONFIG = utils.read_json(CONFIG_PATH) if Path(CONFIG_PATH).exists() else {}
increase_buttons_size = HTML(data="".join(CONFIG.get("save_home_buttons_settings", [])))

# Connect to openBIS
OPENBIS_SESSION, SESSION_DATA = utils.connect_openbis_aiida()
openbis_status_widget = utils.openbis_connection_status_widget(CONFIG, OPENBIS_SESSION)

# Check LLM configuration
LLM_CONFIG_EXISTS = Path(LLM_CONFIG_PATH).exists()
openbis_chatbot = None
if OPENBIS_SESSION and LLM_CONFIG_EXISTS:
    try:
        openbis_chatbot = OpenBISAgent(LLM_CONFIG_PATH)
    except Exception as e:
        print(f"Error initializing chatbot agent: {e}")


In [ ]:
import re
import time
import markdown
from IPython.display import display, HTML, Javascript
import ipywidgets as ipw

# Chat styling - Full width, clean left alignment with header, no horizontal overflow
chat_styles = HTML('''
<style>
  html, body {
    overflow-x: hidden !important;
    width: 100% !important;
  }
  /* Fill to maximum width and eliminate horizontal scrollbars */
  .container, #notebook-container, .jp-Notebook, .jp-Notebook-cell, .cell {
    width: 100% !important;
    max-width: 100% !important;
    margin-left: 0 !important;
    margin-right: 0 !important;
    box-sizing: border-box !important;
  }
  #notebook-container {
    padding-left: 24px !important;
    padding-right: 24px !important;
  }
  /* Remove prompt gutter so code outputs align precisely with markdown text above */
  .prompt, .input_prompt, .output_prompt, 
  .jp-InputPrompt, .jp-OutputPrompt, .jp-Cell-prompt {
    display: none !important;
    min-width: 0 !important;
    width: 0 !important;
    padding: 0 !important;
    margin: 0 !important;
    border: none !important;
  }
  .output_wrapper, .output, .output_area, .output_subarea,
  .jp-OutputArea, .jp-OutputArea-output,
  .cell .inner_cell, .cell .text_cell_render, .cell .rendered_html {
    width: 100% !important;
    max-width: 100% !important;
    margin-left: 0 !important;
    margin-right: 0 !important;
    padding-left: 0 !important;
    padding-right: 0 !important;
    box-sizing: border-box !important;
  }
  .chat-container {
    border: 1px solid #d1d5db;
    border-radius: 12px;
    background: #ffffff;
    padding: 20px 24px;
    height: 520px;
    overflow-y: auto;
    display: flex;
    flex-direction: column;
    gap: 14px;
    box-shadow: inset 0 1px 3px rgba(0, 0, 0, 0.05);
    width: 100%;
    box-sizing: border-box;
  }
  .msg-row-user {
    display: flex;
    justify-content: flex-end;
    margin-bottom: 6px;
    width: 100%;
  }
  .msg-row-agent {
    display: flex;
    justify-content: flex-start;
    margin-bottom: 6px;
    width: 100%;
  }
  .bubble-user {
    background: #1a73e8;
    color: #ffffff;
    padding: 10px 16px;
    border-radius: 16px 16px 2px 16px;
    max-width: 82%;
    font-size: 14px;
    line-height: 1.4;
    box-shadow: 0 1px 2px rgba(0, 0, 0, 0.15);
    word-break: break-word;
  }
  .bubble-agent {
    background: #f8f9fa;
    color: #202124;
    border: 1px solid #e8eaed;
    padding: 14px 18px;
    border-radius: 16px 16px 16px 2px;
    max-width: 94%;
    font-size: 14px;
    line-height: 1.5;
    box-shadow: 0 1px 2px rgba(0, 0, 0, 0.05);
    word-break: break-word;
  }
  .bubble-agent h1, .bubble-agent h2, .bubble-agent h3, .bubble-agent h4, .bubble-agent h5 {
    font-size: 14.5px;
    font-weight: 700;
    color: #1e293b;
    margin: 10px 0 4px 0;
    padding: 0;
    border: none;
    line-height: 1.4;
  }
  .bubble-agent p {
    font-size: 14px;
    line-height: 1.55;
    margin: 6px 0;
    color: #1f2937;
  }
  .bubble-agent ul, .bubble-agent ol {
    margin: 6px 0 8px 0;
    padding-left: 22px;
    font-size: 14px;
    line-height: 1.55;
  }
  .bubble-agent li {
    margin: 2px 0;
  }
  .bubble-agent strong, .bubble-agent b {
    font-weight: 600;
    color: #0f172a;
  }
  .bubble-agent a {
    color: #1a73e8;
    font-weight: 600;
    text-decoration: underline;
  }
  .bubble-agent table {
    border-collapse: collapse;
    width: 100%;
    margin: 12px 0;
    font-size: 13px;
    border: 1px solid #d1d5db;
    border-radius: 6px;
    overflow: hidden;
    box-shadow: 0 1px 3px rgba(0,0,0,0.06);
  }
  .bubble-agent th, .bubble-agent td {
    border: 1px solid #e5e7eb;
    padding: 8px 12px;
    text-align: left;
  }
  .bubble-agent th {
    background-color: #f1f5f9;
    font-weight: 600;
    color: #1e293b;
  }
  .bubble-agent tr:nth-child(even) {
    background-color: #f8fafc;
  }
  .bubble-agent tr:hover {
    background-color: #f1f5f9;
  }
  .bubble-agent code {
    background: #edf2f7;
    padding: 2px 5px;
    border-radius: 4px;
    font-size: 12.5px;
    font-family: monospace;
  }
  .bubble-footer {
    margin-top: 8px;
    padding-top: 6px;
    border-top: 1px solid #e2e8f0;
    font-size: 11.5px;
    color: #64748b;
    display: flex;
    align-items: center;
    gap: 6px;
  }
  .chat-typing-dots {
    display: inline-flex;
    align-items: center;
    gap: 5px;
    margin-left: 2px;
  }
  .chat-typing-dots span {
    width: 7px;
    height: 7px;
    background-color: #1a73e8;
    border-radius: 50%;
    display: inline-block;
    animation: typingBounce 1.4s infinite ease-in-out both;
  }
  .chat-typing-dots span:nth-child(1) { animation-delay: -0.32s; }
  .chat-typing-dots span:nth-child(2) { animation-delay: -0.16s; }
  .chat-typing-dots span:nth-child(3) { animation-delay: 0s; }
  @keyframes typingBounce {
    0%, 80%, 100% { transform: scale(0.6); opacity: 0.35; }
    40% { transform: scale(1.0); opacity: 1; }
  }
  .bubble-agent img {
    max-width: 280px;
    max-height: 220px;
    border-radius: 8px;
    border: 1px solid #e2e8f0;
    background: #ffffff;
    padding: 6px;
    margin: 8px 0 4px 0;
    box-shadow: 0 1px 3px rgba(0, 0, 0, 0.06);
    display: block;
  }
</style>
''')

# Chat state
messages_history = [
    {"role": "agent", "content": "Hello! I am **NanoSurfaces Prime**, your openBIS AI assistant. How can I help you explore your laboratory materials, instruments, preparations, or measurements today?"}
]

def clean_chat_markdown(text: str) -> str:
    if not text:
        return ""
    # Fix tables concatenated without newlines
    text = re.sub(r'\|\s*\|', '|\n|', text)
    # Downscale top-level h1 and h2 markdown headers to ###
    text = re.sub(r'^(#{1,2})\s+(.+)$', r'### \2', text, flags=re.MULTILINE)
    # Normalize LaTeX formulas often output by LLMs: $\text{C}_{64}...$ -> C64...
    text = re.sub(r'\\text\{([A-Za-z0-9]+)\}', r'\1', text)
    text = re.sub(r'\_\{([A-Za-z0-9]+)\}', r'\1', text)
    text = re.sub(r'\^\{([A-Za-z0-9\+\-]+)\}', r'\1', text)
    text = text.replace('$', '')
    return text

def render_messages(is_thinking: bool = False) -> str:
    html_parts = ['<div id="chatbox" class="chat-container">']
    for msg in messages_history:
        if msg["role"] == "user":
            html_parts.append(f'''
            <div class="msg-row-user">
              <div class="bubble-user">👤 <b>You:</b> {msg["content"]}</div>
            </div>
            ''')
        else:
            raw_content = msg["content"]
            fixed_content = clean_chat_markdown(raw_content)
            rendered_md = markdown.markdown(
                fixed_content,
                extensions=['tables', 'fenced_code', 'nl2br']
            )
            elapsed_html = ""
            if msg.get("elapsed") is not None:
                elapsed_html = f'''
                <div class="bubble-footer">
                  <span>⏱️ Answered in {msg["elapsed"]}s</span>
                </div>
                '''
            html_parts.append(f'''
            <div class="msg-row-agent">
              <div class="bubble-agent">
                🤖 <b>NanoSurfaces Prime:</b><br>{rendered_md}
                {elapsed_html}
              </div>
            </div>
            ''')
    if is_thinking:
        html_parts.append('''
        <div class="msg-row-agent">
          <div class="bubble-agent" style="display: inline-flex; align-items: center; gap: 8px; padding: 10px 16px;">
            🤖 <b>NanoSurfaces Prime:</b>
            <span class="chat-typing-dots">
              <span></span><span></span><span></span>
            </span>
          </div>
        </div>
        ''')
    html_parts.append('</div>')
    return "".join(html_parts)

chat_display = ipw.HTML(value=render_messages(), layout=ipw.Layout(width="100%"))

# Prompt input widgets - Flexible width to fill horizontally
prompt_input = ipw.Textarea(
    placeholder="Ask a question (e.g. 'What components are attached to the CreaTec THz-STM?', 'Tell me about substance 704a')...",
    layout=ipw.Layout(flex="1 1 auto", width="auto", min_width="200px", height="75px"),
    continuous_update=True,
)
prompt_input.add_class("chat-prompt-textarea")

send_button = ipw.Button(
    description="Send",
    icon="paper-plane",
    button_style="primary",
    layout=ipw.Layout(width="120px", height="75px", flex="0 0 120px"),
)
send_button.add_class("chat-send-btn")

status_indicator = ipw.HTML(value="")

# Action buttons
clear_button = ipw.Button(
    description="Clear Chat",
    icon="trash",
    button_style="warning",
    layout=ipw.Layout(width="130px", height="40px"),
)

home_button = ipw.Button(
    description="Main Menu",
    icon="home",
    layout=ipw.Layout(width="130px", height="40px"),
)

# JavaScript logic for auto-scrolling and keybindings
KEYBINDINGS_JS = '''
(function() {
    function isChatPrompt(el) {
        if (!el || el.tagName !== 'TEXTAREA') return false;
        if (el.classList.contains('chat-prompt-textarea')) return true;
        if (el.closest && el.closest('.chat-prompt-textarea')) return true;
        if (el.placeholder && el.placeholder.indexOf('Ask a question') !== -1) return true;
        return false;
    }

    function findSendButton() {
        var btnContainer = document.querySelector('.chat-send-btn');
        if (btnContainer) {
            var b = btnContainer.tagName === 'BUTTON' ? btnContainer : btnContainer.querySelector('button');
            if (b) return b;
        }
        var allBtns = document.querySelectorAll('button');
        for (var i = 0; i < allBtns.length; i++) {
            if (allBtns[i].textContent && allBtns[i].textContent.indexOf('Send') !== -1) {
                return allBtns[i];
            }
        }
        return null;
    }

    if (typeof window !== 'undefined' && !window.__openbisChatKeybindingInstalled) {
        window.__openbisChatKeybindingInstalled = true;

        // Prevent Jupyter command mode from capturing keys when typing
        document.addEventListener('focusin', function(e) {
            if (isChatPrompt(e.target)) {
                if (window.Jupyter && window.Jupyter.keyboard_manager) {
                    window.Jupyter.keyboard_manager.disable();
                }
            }
        }, true);

        document.addEventListener('focusout', function(e) {
            if (isChatPrompt(e.target)) {
                if (window.Jupyter && window.Jupyter.keyboard_manager) {
                    window.Jupyter.keyboard_manager.enable();
                }
            }
        }, true);

        // Capture keydown in capturing phase: Enter = Send, Ctrl/Shift/Meta+Enter = Newline
        document.addEventListener('keydown', function(e) {
            if (e.isComposing || e.keyCode === 229) return;
            if (!isChatPrompt(e.target)) return;

            if (e.key === 'Enter') {
                if (e.ctrlKey || e.metaKey || e.shiftKey) {
                    // Multiline newline
                    e.preventDefault();
                    e.stopPropagation();
                    var el = e.target;
                    var start = el.selectionStart;
                    var end = el.selectionEnd;
                    el.value = el.value.substring(0, start) + String.fromCharCode(10) + el.value.substring(end);
                    el.selectionStart = el.selectionEnd = start + 1;
                    el.dispatchEvent(new Event('input', { bubbles: true }));
                    el.dispatchEvent(new Event('change', { bubbles: true }));
                } else {
                    // Send message
                    e.preventDefault();
                    e.stopPropagation();
                    var el = e.target;
                    el.dispatchEvent(new Event('input', { bubbles: true }));
                    el.dispatchEvent(new Event('change', { bubbles: true }));
                    var btn = findSendButton();
                    if (btn) {
                        setTimeout(function() { btn.click(); }, 40);
                    }
                }
            }
        }, true);
    }
})();
'''

def auto_scroll_js():
    return Javascript('''
        var el = document.getElementById("chatbox");
        if (el) { el.scrollTop = el.scrollHeight; }
    ''')

def init_keybindings():
    display(Javascript(KEYBINDINGS_JS))

def on_send_click(b):
    user_text = prompt_input.value.strip()
    if not user_text:
        return
    
    if not openbis_chatbot:
        status_indicator.value = "<span style='color: #d93025; font-weight: bold;'>⚠️ Chatbot is not configured or openBIS not connected.</span>"
        return

    # Add user message
    messages_history.append({"role": "user", "content": user_text})
    prompt_input.value = ""
    # Render with the 3 bouncing dots balloon
    chat_display.value = render_messages(is_thinking=True)
    status_indicator.value = ""
    send_button.disabled = True
    display(auto_scroll_js())

    t0 = time.time()
    try:
        agent_reply = openbis_chatbot.ask_question(user_text)
    except Exception as e:
        agent_reply = f"An error occurred while querying openBIS: {str(e)}"
    elapsed_sec = round(time.time() - t0, 1)

    messages_history.append({
        "role": "agent",
        "content": agent_reply,
        "elapsed": elapsed_sec,
    })
    chat_display.value = render_messages(is_thinking=False)
    send_button.disabled = False
    display(auto_scroll_js())

def on_clear_click(b):
    messages_history.clear()
    messages_history.append({
        "role": "agent",
        "content": "Chat history cleared. How can I help you?"
    })
    if openbis_chatbot:
        openbis_chatbot.clear_history()
    chat_display.value = render_messages()

def on_home_click(b):
    display(Javascript(data='window.location.replace("0_home.ipynb")'))

send_button.on_click(on_send_click)
clear_button.on_click(on_clear_click)
home_button.on_click(on_home_click)

# Layout: Wide, flexible, centered, with input keyboard hint
input_box = ipw.HBox([prompt_input, send_button], layout=ipw.Layout(width="100%", gap="10px", align_items="stretch"))
input_wrapper = ipw.VBox([
    input_box,
    ipw.HTML("<div style='font-size: 11.5px; color: #64748b; margin-top: 4px; padding-left: 2px;'>💡 Press <b>Enter</b> to send &bull; <b>Ctrl+Enter</b> for new line</div>"),
], layout=ipw.Layout(width="100%"))

bottom_bar = ipw.HBox([clear_button, home_button, status_indicator], layout=ipw.Layout(width="100%", gap="12px", align_items="center", margin="12px 0 0 0"))

main_view = ipw.VBox([
    chat_display,
    ipw.HTML("<div style='margin-top: 10px;'></div>"),
    input_wrapper,
    bottom_bar,
], layout=ipw.Layout(width="100%", max_width="100%", margin="0", box_sizing="border-box"))

display(chat_styles)
display(increase_buttons_size)

if not OPENBIS_SESSION:
    display(openbis_status_widget)
elif not LLM_CONFIG_EXISTS:
    display(HTML('''
    <div style="padding: 16px; background: #fef7e0; border: 1px solid #f9ab00; border-radius: 8px; max-width: 800px; margin: 0 auto;">
      <b>⚠️ LLM Chatbot is not configured yet.</b><br>
      Please configure your LLM Provider and API key in <a href="5_setup_chatbot.ipynb" target="_blank">5_setup_chatbot.ipynb</a>.
    </div>
    '''))
else:
    display(main_view)
    init_keybindings()
